### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="garments_worker_productivity",
    dataset_year="2020",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C51S6D",
    download_description=r"""
mkdir -p local-data-warehouse/garments_worker_productivity \
&& wget -P local-data-warehouse/garments_worker_productivity/ https://archive.ics.uci.edu/static/public/597/productivity+prediction+of+garment+employees.zip \
&& unzip local-data-warehouse/garments_worker_productivity/productivity+prediction+of+garment+employees.zip -d local-data-warehouse/garments_worker_productivity/ \
&&  rm local-data-warehouse/garments_worker_productivity/productivity+prediction+of+garment+employees.zip
""",
    # References
    academic_reference_bibtex="""@article{imran2021mining,
    title={Mining the productivity data of the garment industry},
    author={Imran, Abdullah Al and Rahim, Md Shamsur and Ahmed, Tanvir},
    journal={International Journal of Business Intelligence and Data Mining},
    volume={19},
    number={3},
    pages={319--342},
    year={2021},
    publisher={Inderscience Publishers (IEL)}
}
""",
    academic_reference_bibtex_key="imran2021mining",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""
- We fix typos in data entries (e.g., "finishing " becomes "finishing").
- We transform the date to datetime.
- The associated paper conceptualizes the task as an interpretable ML task without consideration of realistic predictive scenarios. Therefore, we define a new predictive ML task.
- We define the decision point in time as the start of the day.
- We set the target variable to "actual_productivity", and use the targeted_productivity as an input feature.
- We do not include all features directly for prediction, because some can't be expected to be available at the decision point in time: "smv", "wip", "over_time", "idle_time", "idle_men". To still keep as much information as possible, we impute the value of the previous recorded working day (by department and team).
- Because the timestamps are irregularly spaced, we additionally include the days passed sind the last recording per department and team.
- Note that we try to keep feature engineering minimalistic and only with the sole purpose to prevent leaks with minimal loss of information.
- It can be expected that feature engineering exposing the non-iid properties of the dataset will be crucial for good predictive performance.
- We cannot fully exclude the possibility of leaks, since not enough information about the date of collection for some features is given.
- We define 30 splits with one day for testing each, resulting in small test sizes of 17-24 samples per split.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="actual_productivity",
    problem_type="regression",
    objective_metric_name="rmse",
    time_on="date"
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

# NOTE: "date", "quarter", "department", "day", "team", "targeted_productivity", "actual_productivity" are safe to keep. For "incentive", "no_of_style_change", "no_of_workers" it's a guess.
lag_cols = ["smv", "wip", "over_time", "idle_time", "idle_men"]

df = pd.read_csv(dataset_mold.path / "garments_worker_productivity.csv")

df.department = df.department.str.strip(" ")
df.date = pd.to_datetime(df.date)

entity_cols = ["department", "team"]
date_col = "date"
lags = (1,)

assert not df.duplicated(["department", "team", "date"]).any(), \
    "Found duplicate rows for the same department-team-date."

df = df.sort_values(list(entity_cols) + [date_col]).reset_index(drop=True)

g = df.groupby(list(entity_cols), sort=False)

prev_date = g[date_col].shift(1)
df["days_since_prev_obs"] = (df[date_col] - prev_date).dt.days

for col in lag_cols:
    for lag in lags:
        df[f"{col}_lag_{lag}"] = g[col].shift(lag)

df = df.drop(columns=lag_cols)

as_cat_type = ["quarter", "department", "day", "team"]
df[as_cat_type] = df[as_cat_type].astype("category")

print("Loaded data shape:", df.shape)
df = df.sort_values(by=date_col).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata

date_col = task_mold.time_on
target_col = task_mold.target_column_name

df = df.sort_values(by=date_col).reset_index(drop=True)

used_in_train = set()
used_in_test = set()
used_data = set()

splits = {}

for split, date in enumerate(sorted(df[date_col].unique())[::-1][:30]):
    train_idx = df[df[date_col] < date].index.tolist()
    test_idx = df[df[date_col] == date].index.tolist()

    splits[split] = {0: [train_idx, test_idx]}

    used_in_train.update(train_idx)
    used_in_test.update(test_idx)
    used_data.update(train_idx)
    used_data.update(test_idx)

    print(f"\n=== Step {split} ===")
    print("Train size:", len(train_idx), "| Test size:", len(test_idx))
    print("Train target mean:", df.loc[train_idx, target_col].mean())
    print("Test target mean:", df.loc[test_idx, target_col].mean())

    assert len(set(train_idx).intersection(set(test_idx)))==0, "Train and test indices overlap!"

print(f"{len(used_data)/df.shape[0]:.4f} of the samples are used.")
print(f"{len(used_in_train)/df.shape[0]:.4f} of the samples are used in training")
print(f"{len(used_in_test)/df.shape[0]:.4f} of the samples are used in testing.")

splits_mold = PredictiveMLSplitsMetadata( 
    splits_comment=r"We define 30 splits with one day for testing each, resulting in small test sizes of 17-24 samples per split. The split with the smallest train sizes uses exactly half of the available samples.",
    splits=splits,
    # We refit every month, so we understand the horizon to be based on months.
    time_horizon=1,
    time_horizon_unit="days",
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()